In [ ]:
import os
import numpy as np
from PIL import Image
import warnings
warnings.filterwarnings('ignore')


In [ ]:
good_dir = "dataset/eggs/good_quality"
poor_dir = "dataset/eggs/poor_quality"
print("Good:", len(os.listdir(good_dir)))
print("Poor:", len(os.listdir(poor_dir)))


In [ ]:
from skimage.color import rgb2gray
from skimage.feature import graycomatrix, graycoprops

IMG_SIZE = 64

def extract_features(path):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img)
    hist_feats = []
    for ch in range(3):
        hist, _ = np.histogram(arr[:, :, ch], bins=8, range=(0, 255))
        hist_feats.extend(hist / hist.sum())
    gray = rgb2gray(arr)
    gray_u8 = (gray * 255).astype(np.uint8)
    glcm = graycomatrix(gray_u8, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    dark_ratio = (gray < (gray.mean() - 0.15)).mean()
    return np.array(hist_feats + [gray.mean(), gray.std(), contrast, energy, dark_ratio])


In [ ]:
X, y = [], []
for label, folder in [(1, good_dir), (0, poor_dir)]:
    for fname in os.listdir(folder):
        X.append(extract_features(os.path.join(folder, fname)))
        y.append(label)
X = np.array(X)
y = np.array(y)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(name, "Accuracy:", round(accuracy_score(y_test, preds) * 100, 2), "%")


In [ ]:
best_model = RandomForestClassifier(n_estimators=100, random_state=42)
best_model.fit(X_train, y_train)


In [ ]:
import pickle
pickle.dump(best_model, open('finalRF_model_egg.sav', 'wb'))
